In [1]:
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
import tensorflow as tf
import tensorflow.keras.backend as K
from tensorflow.keras import (
    layers,
    models,
    callbacks,
    utils,
    metrics,
    losses,
    optimizers,
)

In [ ]:
#load the dataset
ds = load_dataset("Mozilla/flickr30k-transformed-captions", split="test")

dict_keys(['image', 'alt_text', 'sentids', 'split', 'img_id', 'filename', 'original_alt_text'])
['Two people with shaggy hair look at their hands while hanging out in the yard.']


In [ ]:
#preprocess the dataset

ex = ds[100]
#print(len(ds))
#print(len(ds[0]))
#print(ex.keys())
#ex["image"].show()
#print(ex["alt_text"])
#print(ex["filename"])
#print(ex["alt_text"][0]) #that is only the string

def gen():
    for ex in ds:
        img = np.array(ex["image"].convert("RGB"))   # (H, W, 3) uint8
        cap = ex["alt_text"][0]                      # captions
        yield img, cap                             # make just work this funct for 1 iteration when requested (funct is a generator)

tf_ds = tf.data.Dataset.from_generator(
    gen,
    output_signature=(
        tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
        tf.TensorSpec(shape=(), dtype=tf.string),
    ),
)

def preprocess(img, cap):
    img = tf.cast(img, tf.float32) / 127.5 - 1.0
    h, w = tf.shape(img)[0], tf.shape(img)[1]
    s = tf.minimum(h, w)
    img = tf.image.resize_with_crop_or_pad(img, s, s)
    img64  = tf.image.resize(img, [64, 64],  method="area")
    img256 = tf.image.resize(img, [256, 256], method="area")
    return {"img64": img64, "img256": img256, "caption": cap}

BATCH = 32
pipeline = (
    tf_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH)
    .prefetch(tf.data.AUTOTUNE)
)

for img, cap in tf_ds.take(1):
    print("image :", img.shape, img.dtype) 
    print("caption :", cap.numpy())          # cap puts in intelligible letters instead of bytes



image : (500, 333, 3) <dtype: 'uint8'>
caption : b'Two people with shaggy hair look at their hands while hanging out in the yard.'


In [65]:
class Sampling(layers.Layer):
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.random.normal(shape = (batch, dim))
        return z_mean + tf.exp(0.5*z_log_var)*epsilon  # basically sample a single point from the distibution 

In [ ]:
#define the VAE model: encoder -> decoder , GAN

encoder_input = layers.Input(shape = (64,64,3),name = "encoder_input")
x = layers.Conv2D(64, (9,9), strides = 2, padding = "same")(encoder_input)
x = layers.GroupNormalization()(x)
x = layers.LeakyReLU(0.3)(x)
x = layers.Conv2D(128, (3,3), strides = 2, padding = "same")(x)
x = layers.GroupNormalization()(x)
x = layers.LeakyReLU(0.3)(x)
x = layers.Conv2D(256, (3,3), strides = 2, padding = "same")(x)
x = layers.GroupNormalization()(x)
x = layers.LeakyReLU(0.3)(x)
x = layers.Conv2D(512, (3,3), strides = 2, padding = "same")(x)
x = layers.GroupNormalization()(x)
x = layers.LeakyReLU(0.3)(x)
shape_bf_flat = np.shape(x)
t_, x_, y_, z_ = shape_bf_flat # shape in each dimension
x = layers.Flatten()(x)
z_mean = layers.Dense(512, name = "z_mean")(x)
z_log_var = layers.Dense(512, name = "z_log_var")(x)
encoder_output = Sampling()([z_mean, z_log_var])
encoder = models.Model(encoder_input,[z_mean, z_log_var, encoder_output], name = "encoder")


decoder_input = layers.Input(shape = (512,), name = "decoder_input")
x = layers.Dense(x_*y_*z_)(decoder_input)
x = layers.Reshape((4,4,512))(x)
x = layers.UpSampling2D(size = 2, interpolation = "nearest")(x)
x = layers.Conv2D(256 , (3,3), strides = 1 , padding = "same")(x)
x = layers.GroupNormalization()(x)
x = layers.LeakyReLU(0.3)(x)
x = layers.UpSampling2D(size = 2, interpolation = "nearest")(x)
x = layers.Conv2D(128 , (3,3), strides = 1 , padding = "same")(x)
x = layers.GroupNormalization()(x)
x = layers.LeakyReLU(0.3)(x)
x = layers.UpSampling2D(size = 2, interpolation = "nearest")(x)
x = layers.Conv2D(64 , (3,3), strides = 1 , padding = "same")(x)
x = layers.GroupNormalization()(x)
x = layers.LeakyReLU(0.3)(x)
x = layers.UpSampling2D(size = 2, interpolation = "nearest")(x)
x = layers.Conv2D(32 , (3,3), strides = 1 , padding = "same")(x)
x = layers.GroupNormalization()(x)
x = layers.LeakyReLU(0.3)(x)
decoder_output = layers.Conv2D(3, 3, strides = 1 , padding = "same", activation = "tanh")(x)
decoder = models.Model(decoder_input, decoder_output , name = "decoder")

encoder.summary()
decoder.summary()


Model: "encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 64, 64, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_110 (Conv2D) │ (None, 32, 32,    │     15,616 │ encoder_input[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ group_normalizatio… │ (None, 32, 32,    │        128 │ conv2d_110[0][0]  │
│ (GroupNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_104     │ (None, 32, 32,    │          0 │ group_normalizat… │
│ (LeakyReLU)         │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_111 (Conv2D) │ (None, 16, 16,    │     73,856 │ leaky_re_lu_104[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ group_normalizatio… │ (None, 16, 16,    │        256 │ conv2d_111[0][0]  │
│ (GroupNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_105     │ (None, 16, 16,    │          0 │ group_normalizat… │
│ (LeakyReLU)         │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_112 (Conv2D) │ (None, 8, 8, 256) │    295,168 │ leaky_re_lu_105[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ group_normalizatio… │ (None, 8, 8, 256) │        512 │ conv2d_112[0][0]  │
│ (GroupNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_106     │ (None, 8, 8, 256) │          0 │ group_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_113 (Conv2D) │ (None, 4, 4, 512) │  1,180,160 │ leaky_re_lu_106[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ group_normalizatio… │ (None, 4, 4, 512) │      1,024 │ conv2d_113[0][0]  │
│ (GroupNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_107     │ (None, 4, 4, 512) │          0 │ group_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_20          │ (None, 8192)      │          0 │ leaky_re_lu_107[… │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_mean (Dense)      │ (None, 512)       │  4,194,816 │ flatten_20[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ z_log_var (Dense)   │ (None, 512)       │  4,194,816 │ flatten_20[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sampling_17         │ (None, 512)       │          0 │ z_mean[0][0],     │
│ (Sampling)          │                   │            │ z_log_var[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 9,956,352 (37.98 MB)

 Trainable params: 9,956,352 (37.98 MB)

 Non-trainable params: 0 (0.00 B)

Model: "decoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ decoder_input (InputLayer)      │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 8192)           │     4,202,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_9 (Reshape)             │ (None, 4, 4, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_6 (UpSampling2D)  │ (None, 8, 8, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_114 (Conv2D)             │ (None, 8, 8, 256)      │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_108         │ (None, 8, 8, 256)      │           512 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_108 (LeakyReLU)     │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_7 (UpSampling2D)  │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_115 (Conv2D)             │ (None, 16, 16, 128)    │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_109         │ (None, 16, 16, 128)    │           256 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_109 (LeakyReLU)     │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_8 (UpSampling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_116 (Conv2D)             │ (None, 32, 32, 64)     │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_110         │ (None, 32, 32, 64)     │           128 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_110 (LeakyReLU)     │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ up_sampling2d_9 (UpSampling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_117 (Conv2D)             │ (None, 64, 64, 32)     │       165,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_111         │ (None, 64, 64, 32)     │            64 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ leaky_re_lu_111 (LeakyReLU)     │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_118 (Conv2D)             │ (None, 64, 64, 3)      │           867 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,918,979 (22.58 MB)

 Trainable params: 5,918,979 (22.58 MB)

 Non-trainable params: 0 (0.00 B)

In [15]:
#define the train function for VAE -> train step , losses 

In [16]:
#train the model

In [17]:
#define the diffusion model and text encoder

In [18]:
#define the train function for diffusion and embedding model -> train step , losses ...

In [19]:
#train the model

In [ ]:
#define the function for generation